In [10]:
from src.tasks import TASKS

sample = TASKS["multiply"].sample()

# Print all field names and values on the Instance object
print(vars(sample))

{'prompt': '41*62', 'correct_trace': '2*41=82 60*41=2460 82+2460=2542', 'wrong_trace': '2*41=81 60*41=5870 81+5870=5951', 'gold': '2542'}


In [2]:
TASKS

{'word_index': Task(name='word_index', chars='abcdefghijklmnopqrstuvwxyz0123456789 ;:\n', block_size=64, max_new_tokens=35, sample=<function _sample_word_index at 0x711bfc65f560>, chance_acc=0.15652, ceiling_acc=0.89259, description='report the index of a queried letter; trace enumerates (i, char)', answer_pattern='\\d+', bayes_prob=<function _word_index_bayes at 0x711bfc528400>, tokenizer=<src.tasks.CharTokenizer object at 0x711bfc69ad80>),
 'sort_letters': Task(name='sort_letters', chars='abcdefghijklmnopqrstuvwxyz :>\n', block_size=72, max_new_tokens=60, sample=<function _sample_sort_letters at 0x711bfc5284a0>, chance_acc=5e-05, ceiling_acc=1.0, description='alphabetize a word; trace is selection sort (remainder > min)', answer_pattern='[a-z]+', bayes_prob=None, tokenizer=<src.tasks.CharTokenizer object at 0x711bfc69a210>),
 'multiply': Task(name='multiply', chars='0123456789 :*+=\n', block_size=64, max_new_tokens=48, sample=<function _sample_multiply at 0x711bfc528680>, chance_acc=

In [3]:
import random
import string
from dataclasses import dataclass, field

@dataclass(frozen=True)
class Instance:
    prompt: str
    correct_trace: str
    wrong_trace: str
    gold: str
def _sample_sort_letters() -> Instance:
    L = random.randint(7, 10)
    letters = random.sample(string.ascii_lowercase, L)
    word = "".join(letters)
    gold = "".join(sorted(letters))

    remaining, steps = list(letters), []
    while len(remaining) > 1:
        chosen = min(remaining)
        steps.append(f"{''.join(remaining)}>{chosen}")
        remaining.remove(chosen)
    correct = " ".join(steps)

    remaining, steps = list(letters), []
    while len(remaining) > 1:
        correct_choice = min(remaining)
        wrong_choices = [x for x in remaining if x != correct_choice]
        chosen = random.choice(wrong_choices)

        steps.append(f"{''.join(remaining)}>{chosen}")
        remaining.remove(chosen)
    wrong = " ".join(steps)

    return Instance(word, correct, wrong, gold)

In [4]:
vars(_sample_sort_letters())

{'prompt': 'qvyuswlix',
 'correct_trace': 'qvyuswlix>i qvyuswlx>l qvyuswx>q vyuswx>s vyuwx>u vywx>v ywx>w yx>x',
 'wrong_trace': 'qvyuswlix>s qvyuwlix>v qyuwlix>q yuwlix>w yulix>l yuix>u yix>y ix>x',
 'gold': 'ilqsuvwxy'}

In [5]:
import random
from collections import deque
from typing import Dict, List, Tuple

def _sample_graph_path() -> Instance:
    n = random.randint(4, 6)
    nodes = list(range(n))

    # Random sparse undirected graph
    possible = [(i, j) for i in nodes for j in nodes if i < j]
    random.shuffle(possible)
    n_edges = random.randint(n - 1, min(len(possible), n + 2))
    edges = possible[:n_edges]

    adj = {i: [] for i in nodes}
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)

    s, t = random.sample(nodes, 2)

    # Prompt construction
    edge_str = " ".join(f"{u}-{v}" for u, v in edges)
    prompt = f"{edge_str};{s}>{t}"

    dist, correct_layers = _bfs_shortest(adj, s, t)
    gold = str(dist)
    correct = " ".join(correct_layers)

    # Fix: Match sequence length and initial anchor (s=0) to prevent heuristic shortcuts
    k = len(correct_layers)
    other_nodes = [node for node in nodes if node != s]

    if k - 1 <= len(other_nodes):
        fake_other_nodes = random.sample(other_nodes, k - 1)
    else:
        fake_other_nodes = random.choices(nodes, k=k - 1)

    fake_nodes = [s] + fake_other_nodes
    fake_dists = [0] + [random.randint(1, n) for _ in range(k - 1)]

    wrong_layers = [f"{node}={d}" for node, d in zip(fake_nodes, fake_dists)]

    # Collision guard: Ensure wrong trace is never identical to correct trace
    if wrong_layers == correct_layers:
        fake_dists[-1] = (fake_dists[-1] + 1) % (n + 1)
        wrong_layers = [f"{node}={d}" for node, d in zip(fake_nodes, fake_dists)]

    wrong = " ".join(wrong_layers)

    return Instance(prompt, correct, wrong, gold)

vars(_sample_graph_path())


NameError: name '_bfs_shortest' is not defined

In [9]:
def _sample_count_char() -> Instance:
    L = random.randint(6, 14)
    alphabet = string.ascii_lowercase[:3]
    word = "".join(random.choice(alphabet) for _ in range(L))
    query = random.choice(word)
    prompt = f"{word};{query}"
    running, counts = 0, []
    for ch in word:
        running += int(ch == query)
        counts.append(running)
    correct = " ".join(f"{ch}" for ch, n in zip(word, counts))

    # Envelope-preserving corruption: same non-decreasing, step<=1 sequence,
    # attributed to the wrong position (off-by-one misattribution).
    wrong_counts = [0] + counts[:-1]
    if wrong_counts == counts:
        # Degenerate collision guard (only possible if counts is constant, which
        # cannot happen since `query` is guaranteed to occur at least once).
        wrong_counts = counts[1:] + [counts[-1]]
    wrong = " ".join(f"{"x"}" for ch, n in zip(word, wrong_counts))

    assert wrong_counts != counts, "Wrong trace collided with correct trace"
    return Instance(prompt, correct, wrong, str(running))

vars(_sample_count_char())

{'prompt': 'bababcbcc;c',
 'correct_trace': 'b a b a b c b c c',
 'wrong_trace': 'x x x x x x x x x',
 'gold': '3'}

In [ ]:
import random

DIGITS = "0123456789"


def make_product_string(a_range=(10, 99), b_range=(10, 99)):
    a = random.randint(*a_range)
    b = random.randint(*b_range)
    ans = a * b
    return str(ans), a, b, ans


def sample_bayes_irrelevant_instance():
    """Mirrors Definition 1 exactly: draw W (here, a product's digit string),
    draw I* uniformly at random among occurrence positions of some digit,
    independent of W given its length; C is read off from I*."""
    W, a, b, ans = make_product_string()
    L = len(W)

    # pick a query digit that actually occurs in W (so there's ambiguity to test)
    present_digits = list(set(W))
    C = random.choice(present_digits)
    matching_positions = [i for i, ch in enumerate(W) if ch == C]

    # I* drawn uniformly among matches -- the genuine, irreducible randomness
    I_star = random.choice(matching_positions)

    prompt = f"{W};{C}"
    gold = str(I_star)
    return prompt, gold, W, C, I_star, matching_positions, a, b, ans


def correct_trace(W):
    """Deterministic re-encoding of W -- carries no info beyond W itself,
    exactly like the paper's Rgood."""
    return " ".join(f"{i}{ch}" for i, ch in enumerate(W))


def corrupted_trace(W):
    """Shape-preserving but fully decorrelated: random permutation of
    positions, i.i.d. random digits -- exactly like the paper's Rbad.
    Independent of W, of I*, and of itself across positions."""
    L = len(W)
    perm = list(range(L))
    random.shuffle(perm)
    rand_digits = [random.choice(DIGITS) for _ in range(L)]
    return " ".join(f"{perm[i]}{rand_digits[i]}" for i in range(L))


def make_example(mode="think_correct"):
    prompt, gold, W, C, I_star, matches, a, b, ans = sample_bayes_irrelevant_instance()

    if mode == "think_correct":
        trace = correct_trace(W)
    elif mode == "think_wrong":
        trace = corrupted_trace(W)
    elif mode == "none":
        return prompt, f" {gold}\n"
    else:
        raise ValueError(f"unknown mode {mode!r}")

    target = f" {trace} : {gold}\n"
    return prompt, target


def build_corpus(n, mode="think_correct"):
    return [make_example(mode) for _ in range(n)]
# ============================================================
def sample_compute_then_search_instance():
    a = random.randint(10, 99)
    b = random.randint(10, 99)
    ans = a * b
    W = str(ans)
    present_digits = list(set(W))
    C = random.choice(present_digits)
    matches = [i for i, ch in enumerate(W) if ch == C]
    I_star = random.choice(matches)
    prompt = f"{a}*{b};{C}"
    gold = str(I_star)
    return prompt, gold, a, b, ans, W, C


if __name__ == "__main__":
    random.seed(10)
    p, g, W, C, Is, matches, a, b, ans = sample_bayes_irrelevant_instance()
    print("Transplant task example:")
    print(f"  prompt={p!r}  gold={g!r}  (W={W}, digit={C}, matches at {matches})")
    print(f"  correct trace: {correct_trace(W)!r}")
    print(f"  corrupted trace: {corrupted_trace(W)!r}")

    p2, g2, a2, b2, ans2, W2, C2 = sample_compute_then_search_instance()
    print("\nContrast (compute-then-search) task example:")
    print(f"  prompt={p2!r}  gold={g2!r}  (a={a2}, b={b2}, ans={ans2}, digit={C2})")

Transplant task example:
  prompt='1162;2'  gold='3'  (W=1162, digit=2, matches at [3])
  correct trace: '01 11 26 32'
  corrupted trace: '27 14 32 00'

Contrast (compute-then-search) task example:
  prompt='76*72;5'  gold='0'  (a=76, b=72, ans=5472, digit=5)


In [3]:
1162*2

2324